# Master Benchmarking Suite: 12 Evolutionary Architectures (Model 1 -> Model 12)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doantrongthai/flood_mamba_edge/blob/main/screening_aquamam_models.ipynb)

This notebook automates sequential multi-dataset evaluation across all 12 systematic evolutions of **AquaMamNet**:
- **Model 1 (`model1`)**: Spatial Boundary-Gated PixelShuffle Decoder (`SpatialBoundaryPixelShuffle`)
- **Model 2 (`model2`)**: Scale-Aware Multi-Scale Mamba Fusion (`ScaleAwareMambaFusion`)
- **Model 3 (`model3`)**: Dynamic Omni-Directional Strip Convolution (`OmniASDConv`)
- **Model 4 (`model4`)**: Inverted Residual Strip Bottleneck (`InvResASDBlock`)
- **Model 5 (`model5`)**: Differentiable Laplacian Boundary Guidance (`DifferentiableLaplacianGuidance`)
- **Model 6 (`model6`)**: Multi-Scale Dilated Refinement (`MultiDilatedSGR`)
- **Model 7 (`model7`)**: Dual-Path Local-Global Fusion (`DualPathMambaFusion`)
- **Model 8 (`model8`)**: Boundary Enhancement Module Integration (`BEM-Aqua`)
- **Model 9 (`model9`)**: Scale-Aware + Spatial Boundary Gating Synergy
- **Model 10 (`model10`)**: Omni-Strip + Spatial Boundary Gating Synergy
- **Model 11 (`model11`)**: Tri-Synergy (Omni-Strip + Scale-Aware + Spatial Boundary Gating)
- **Model 12 (`model12`)**: Master Super-Aqua with Lite Deep Supervision & Laplacian Guidance

### Default Experimental Setup
- **Resolution**: $128 \times 128$
- **Batch Size**: 4
- **Epochs**: 100
- **Loss Function**: `lovasz_focal`
- **Learning Rate**: 0.001
- **Seed**: 42
- **Datasets**: `floodkaggle`, `aifloodsense`

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Repository Setup
import os
if not os.path.exists('flood_mamba_edge'):
    !git clone https://github.com/doantrongthai/flood_mamba_edge.git
%cd flood_mamba_edge
!git pull origin main

In [ ]:
# 3. Dependency Installation
!pip install -q -r requirements.txt
!pip install -q gdown tabulate

# Optional Hardware CUDA acceleration kernel (with pure PyTorch fallback guaranteed)
!pip install -q --no-build-isolation mamba-ssm causal-conv1d || echo 'Using pure PyTorch fallback for seamless execution.'

In [ ]:
# 4. Architecture Registry & Standard Configuration
SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 100
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1
DATASETS = ["floodkaggle", "aifloodsense"]

# 12 Novel Evolutionary Architectures
MODELS_TO_BENCHMARK = [
    "model1",   # Spatial Boundary-Gated PixelShuffle
    "model2",   # Scale-Aware Multi-Scale Mamba
    "model3",   # Dynamic Omni-Directional Strip Conv
    "model4",   # Inverted Residual Strip Bottleneck
    "model5",   # Differentiable Laplacian Boundary Guidance
    "model6",   # Multi-Scale Dilated Refinement (1, 2, 4)
    "model7",   # Dual-Path Local-Global Hybrid Fusion
    "model8",   # Boundary Enhancement Module Integration
    "model9",   # Scale-Aware + Spatial Boundary Gating
    "model10",  # Omni-Strip + Spatial Boundary Gating
    "model11",  # Tri-Synergy (Omni + Scale + Spatial Gate)
    "model12",  # Master Super-Aqua with Deep Supervision
]

print(f"Total architectures registered: {len(MODELS_TO_BENCHMARK)}")
print(f"Parameters: Batch={BATCH_SIZE}, Size={IMAGE_SIZE}, Epochs={EPOCHS}, Loss={LOSS_NAME}, LR={LR}, Seed={SEED}")

In [ ]:
# 5. Pre-Download Datasets (Fast verification)
import sys, subprocess
for ds in DATASETS:
    print(f"Verifying dataset: {ds}...")
    subprocess.run([sys.executable, "main.py", "--dataset", ds, "--download", "--epochs", "1", "--batch_size", "2", "--model", "model1", "--size", "128"], check=False)

In [ ]:
# 6. Master Automated Training & Evaluation Loop
import time, re

results = []
total_runs = len(MODELS_TO_BENCHMARK) * len(DATASETS)
run_idx = 0

print(f"Starting master execution: {total_runs} runs planned.")

for m_idx, model_name in enumerate(MODELS_TO_BENCHMARK):
    for ds in DATASETS:
        run_idx += 1
        print(f"\n{'#'*85}")
        print(f"[{run_idx}/{total_runs}] BENCHMARKING: {model_name.upper()} on {ds.upper()} (Seed {SEED})")
        print(f"{'#'*85}")
        
        cmd = [
            sys.executable, "main.py",
            "--dataset", ds,
            "--model", model_name,
            "--num_classes", str(NUM_CLASSES),
            "--loss", LOSS_NAME,
            "--size", str(IMAGE_SIZE),
            "--epochs", str(EPOCHS),
            "--batch_size", str(BATCH_SIZE),
            "--lr", str(LR),
            "--seed", str(SEED)
        ]
        
        start_t = time.time()
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        
        miou_val = None
        dice_val = None
        prec_val = None
        rec_val = None
        biou_val = None
        best_val_loss = None
        
        for line in proc.stdout:
            line_str = line.strip()
            if any(k in line_str for k in ["Epoch", "mIoU:", "TESTING", "Best", "Val:"]):
                print(line_str)
            if "mIoU:" in line_str and "Dice:" in line_str:
                m_iou = re.search(r'mIoU:\s*\$?([0-9.]+)', line_str)
                m_dice = re.search(r'Dice:\s*\$?([0-9.]+)', line_str)
                m_prec = re.search(r'Prec:\s*\$?([0-9.]+)', line_str)
                m_rec = re.search(r'Rec:\s*\$?([0-9.]+)', line_str)
                m_biou = re.search(r'B-IoU:\s*\$?([0-9.]+)', line_str)
                if m_iou: miou_val = float(m_iou.group(1))
                if m_dice: dice_val = float(m_dice.group(1))
                if m_prec: prec_val = float(m_prec.group(1))
                if m_rec: rec_val = float(m_rec.group(1))
                if m_biou: biou_val = float(m_biou.group(1))
            if "Best Val Loss:" in line_str:
                m_loss = re.search(r'Best Val Loss:\s*([0-9.]+)', line_str)
                if m_loss: best_val_loss = float(m_loss.group(1))
                
        proc.wait()
        elapsed = time.time() - start_t
        
        entry = {
            'model': model_name,
            'dataset': ds,
            'mIoU': miou_val,
            'Dice': dice_val,
            'Precision': prec_val,
            'Recall': rec_val,
            'B_IoU': biou_val,
            'val_loss': best_val_loss,
            'time_sec': elapsed
        }
        results.append(entry)
        print(f"--> [{model_name} on {ds}] Finished in {elapsed:.1f}s | mIoU={miou_val}, Dice={dice_val}, B-IoU={biou_val}")

print("\n\u2713 All benchmarking runs finished successfully!")

In [ ]:
# 7. Generate Paper-Ready Results Tables
import pandas as pd
from tabulate import tabulate

df = pd.DataFrame(results)
print("=== MASTER SUMMARY TABLE ===")
print(tabulate(df, headers='keys', tablefmt='github', floatfmt='.4f'))

# Pivot table for mIoU
if 'dataset' in df.columns and 'mIoU' in df.columns:
    piv = df.pivot(index='model', columns='dataset', values='mIoU')
    print("\n=== mIoU COMPARISON ACROSS DATASETS ===")
    print(tabulate(piv, headers='keys', tablefmt='github', floatfmt='.4f'))

# Save CSV
df.to_csv("aquamam_evolution_results.csv", index=False)
print("\nSaved results to aquamam_evolution_results.csv")